# Nutrient Mapping
Maps Project 2 food prices to micronutrients from min_cost_data using fuzzy matching.

## Step 1 — Install & Import

In [26]:
import pandas as pd
import numpy as np
import difflib
import warnings
warnings.filterwarnings('ignore')

## Step 2 — Load Data

In [28]:
# Your foods with prices
foods = pd.read_csv('foods_with_fcd_id.csv')
print('Foods shape:', foods.shape)

# Nutrients from min_cost_data
# We read the 'nutrients' sheet directly
nutrients = pd.read_excel('min_cost_data(1).xlsx', sheet_name='nutrients', engine='openpyxl')
print('Nutrients shape:', nutrients.shape)

Foods shape: (176, 8)
Nutrients shape: (2750, 67)


In [29]:
# # Load nutrients from Excel
# nutrients = pd.read_excel('min_cost_data(1).xlsx', sheet_name='nutrients', engine='openpyxl')
# print('Nutrients shape:', nutrients.shape)
# nutrients.head()

## Step 3 — Fuzzy Match Food Names

In [31]:
food_names = foods['Food'].str.lower().str.strip().tolist()
nut_desc = nutrients['Ingredient description'].tolist()
nut_desc_lower = [d.lower().strip() for d in nut_desc]
desc_map = dict(zip(nut_desc_lower, nut_desc))

matches = []
for food in food_names:
    # difflib.get_close_matches finds the best "look-alike" name
    best_match_list = difflib.get_close_matches(food, nut_desc_lower, n=1, cutoff=0.6)
    
    if best_match_list:
        best_match = desc_map[best_match_list[0]]
        score = 80 # difflib doesn't give a 0-100 score easily, so we set a baseline
    else:
        best_match = "Manual Review Required"
        score = 0

    matches.append({
        'Food': food,
        'matched_to': best_match,
        'score': score
    })

match_df = pd.DataFrame(matches)

In [19]:
# # Review low confidence matches — fix these manually in fuzzy_matches.csv if needed
# low_confidence = match_df[match_df['score'] < 80]
# print(f'Low confidence matches: {len(low_confidence)}')
# low_confidence

Low confidence matches: 1


,p2_food,matched_to,score
34,avacado,"Avocados, raw, all commercial varieties",77


In [32]:
manual_fixes = {
    "white flour": "Flour, wheat, all-purpose, enriched, bleached",
    "rice long grain ": "Rice, white, long-grain, regular, raw, enriched",
    "spaghetti ": "Pasta, dry, enriched",
    "white bread": "Bread, white, commercially prepared (includes soft bread crumbs)",
    "whole wheat bread ": "Bread, whole-wheat, commercially prepared",
    "chocolate chip cookie ": "Cookies, chocolate chip, prepared from recipe, made with butter",
    "ground chuck beef ": "Beef, ground, 80% lean meat / 20% fat, raw",
    "ground beef ": "Beef, ground, 80% lean meat / 20% fat, raw",
    "stew beef ": "Beef, round, bottom round, roast, separable lean only, raw",
    "steak, round": "Beef, round, eye of round, roast, separable lean only, raw",
    "steak, sirloin": "Beef, loin, top sirloin steak, separable lean only, raw",
    "chicken breast, boneless ": "Chicken, broilers or fryers, breast, meat only, raw",
    "chicken legs, bone-in": "Chicken, broilers or fryers, drumstick, meat and skin, raw",
    "butter, stick ": "Butter, salted",
    "bacon": "Pork, cured, bacon, unprepared",
    "pork chops ": "Pork, fresh, loin, top loin (chops), boneless, separable lean only, raw",
    "ham": "Pork, cured, ham, whole, separable lean only, roasted",
    "grade A eggs ": "Egg, whole, raw, fresh",
    "whole milk ": "Milk, whole, 3.25% milkfat, with added vitamin D",
    "low-fat milk": "Milk, lowfat, fluid, 1% milkfat, with added vitamin A and vitamin D",
    "yogurt, whole-fat plain ": "Yogurt, plain, whole milk, 8 grams protein per 8 ounce",
    "american cheese ": "Cheese, pasteurized process, American, fortified with vitamin D",
    "white sugar ": "Sugars, granulated",
    "ground coffee ": "Beverages, coffee, instant, regular, powder",
    "blackeye peas, dried ": "Cowpeas, common (blackeyes, crowder, southern), mature seeds, raw",
    "navy beans, canned": "Beans, navy, mature seeds, canned, no salt added",
    "avocado": "Avocados, raw, all commercial varieties",
    "green cabbage ": "Cabbage, raw",
    "red cabbage ": "Cabbage, red, raw",
    "artichoke, fresh ": "Artichokes, (globe or french), raw",
    "broccoli heads, fresh ": "Broccoli, raw",
    "carrots, whole, fresh ": "Carrots, raw",
    "collard greens, fresh ": "Collard greens, raw",
    "corn, fresh ": "Corn, sweet, yellow, raw",
    "cucumbers, fresh ": "Cucumber, with peel, raw",
    "green beans, fresh": "Beans, snap, green, raw",
    "lettuce, iceberg": "Lettuce, iceberg (includes crisphead types), raw",
    "potatoes, fresh ": "Potatoes, flesh and skin, raw",
    "potatoes, french fries": "Potatoes, french fried, all types, salt added in processing, frozen",
    "tomatoes, grape and cherry": "Tomatoes, red, ripe, raw, year round average",
    "zucchini, fresh ": "Squash, summer, zucchini, includes skin, raw",
    "apples": "Apples, raw, with skin",
    "apple juice ": "Apple juice, canned or bottled, unsweetened, without added ascorbic acid",
    "applesauce": "Applesauce, canned, unsweetened, without added ascorbic acid",
    "bananas, fresh ": "Bananas, raw",
    "cantaloupe, fresh ": "Melons, cantaloupe, raw",
    "clementines, fresh ": "Tangerines, (mandarin oranges), raw",
    "grape juice ": "Grape juice, canned or bottled, unsweetened, without added ascorbic acid",
    "oranges": "Oranges, raw, all commercial varieties",
    "orange, frozen juice concentrate": "Orange juice, frozen concentrate, unsweetened, undiluted",
    "peaches, fresh ": "Peaches, raw",
    "pineapple juice": "Pineapple juice, canned or bottled, unsweetened, without added ascorbic acid",
    "plum (prunes), dried": "Plums, dried (prunes), uncooked",
    "raisins": "Raisins, seedless"
}

# Apply fixes
for food, fix in manual_fixes.items():
    # We clean the food name just like we did in Step 3
    food_key = food.lower().strip()
    match_df.loc[match_df['Food'].str.lower().str.strip() == food_key, 'matched_to'] = fix
    match_df.loc[match_df['Food'].str.lower().str.strip() == food_key, 'score'] = 100

print(f'Applied manual fixes. Matches ready for merge.')

Applied manual fixes. Matches ready for merge.


## Step 4 — Merge Prices with Nutrients

In [33]:
# Add matched name to foods
foods['matched_name'] = foods['Food'].str.lower().str.strip().map(
    dict(zip(match_df['Food'].str.lower().str.strip(), match_df['matched_to']))
)

# Merge
merged = foods.merge(
    nutrients,
    left_on='matched_name',
    right_on='Ingredient description',
    how='inner'
)

print(f'Successfully merged: {len(merged)} foods')
merged[['Food', 'Price', 'matched_name']].head(10)

Successfully merged: 151 foods


,Food,Price,matched_name
0,rice long grain,0.970,"Rice, white, long-grain, regular, raw, enriched"
1,spaghetti,1.475,"Pasta, dry, enriched"
2,white bread,1.888,"Bread, white, commercially prepared (includes ..."
3,whole wheat bread,2.451,"Bread, whole-wheat, commercially prepared"
4,ground chuck beef,4.644,"Beef, ground, 80% lean meat / 20% fat, raw"
5,ground beef,4.791,"Beef, ground, 80% lean meat / 20% fat, raw"
6,lean ground beef,6.393,"Lamb, ground, raw"
7,bacon,6.808,"Pork, cured, bacon, unprepared"
8,pork chops,4.573,"Pork, fresh, loin, top loin (chops), boneless,..."
9,ham,5.484,"Pork, cured, ham, whole, separable lean only, ..."


## Step 5 — Normalize Prices to Per-100g

In [34]:
# Add matched name to foods
foods['matched_name'] = foods['Food'].str.lower().str.strip().map(
    dict(zip(match_df['Food'].str.lower().str.strip(), match_df['matched_to']))
)

# Merge
merged = foods.merge(
    nutrients,
    left_on='matched_name',
    right_on='Ingredient description',
    how='inner'
)

print(f'Successfully merged: {len(merged)} foods')
merged[['Food', 'Price', 'matched_name']].head(10)

Successfully merged: 151 foods


,Food,Price,matched_name
0,rice long grain,0.970,"Rice, white, long-grain, regular, raw, enriched"
1,spaghetti,1.475,"Pasta, dry, enriched"
2,white bread,1.888,"Bread, white, commercially prepared (includes ..."
3,whole wheat bread,2.451,"Bread, whole-wheat, commercially prepared"
4,ground chuck beef,4.644,"Beef, ground, 80% lean meat / 20% fat, raw"
5,ground beef,4.791,"Beef, ground, 80% lean meat / 20% fat, raw"
6,lean ground beef,6.393,"Lamb, ground, raw"
7,bacon,6.808,"Pork, cured, bacon, unprepared"
8,pork chops,4.573,"Pork, fresh, loin, top loin (chops), boneless,..."
9,ham,5.484,"Pork, cured, ham, whole, separable lean only, ..."


## Step 6 — Build Matrix A and Price Vector p

In [35]:
unit_to_grams = {
    'pound': 453.592, 'pound ': 453.592, 'lb': 453.592,
    'oz': 28.3495, 'gallon': 3785.41, 'egg': 50.0,
}

def price_per_100g(row):
    unit = str(row['Units']).strip().lower()
    qty = row['Quantity']
    price = row['Price']
    grams = unit_to_grams.get(unit, None)
    if grams is None or qty == 0:
        return None
    return (price / (qty * grams)) * 100

merged['price_per_100g'] = merged.apply(price_per_100g, axis=1)
merged = merged.dropna(subset=['price_per_100g'])
merged = merged.sort_values('price_per_100g').drop_duplicates(subset='Food')

print(f'Foods with valid prices: {len(merged)}')

Foods with valid prices: 151


In [36]:
non_nutrient_cols = ['Food', 'Quantity', 'Units', 'Price', 'Year', 'FCD_id',
                     'FDC_match', 'FDC_type', 'matched_name', 'ingred_code',
                     'Ingredient description', 'price_per_100g']

nutrient_cols = [c for c in merged.columns if c not in non_nutrient_cols]

# This is your final matrix
A = merged.set_index('Food')[nutrient_cols].fillna(0)

# This is your final price vector
p = merged.set_index('Food')['price_per_100g']

print('Matrix A shape (foods x nutrients):', A.shape)
A.head()

Matrix A shape (foods x nutrients): (151, 65)


,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic Acid,Linolenic Acid,Stearidonic acid,...,Vitamin B12,"Vitamin B-12, added",Vitamin B6,Vitamin C,Vitamin D,Vitamin E,"Vitamin E, added",Vitamin K,Water,Zinc
Food,,,,,,,,,,,,,,,,,,,,,
Watermelon,0.001,0.001,0.000,0.008,0.000,0.006,0.037,0.050,0.000,0.0,...,0.00,0.0,0.045,8.1,0.0,0.05,0.0,0.1,91.45,0.10
low-fat milk,0.027,0.029,0.091,0.287,0.017,0.126,0.250,0.030,0.004,0.0,...,0.47,0.0,0.037,0.0,1.2,0.01,0.0,0.1,89.92,0.42
whole milk,0.075,0.077,0.297,0.829,0.000,0.365,0.812,0.120,0.075,0.0,...,0.45,0.0,0.036,0.0,1.3,0.07,0.0,0.3,88.13,0.37
Bananas,0.001,0.002,0.002,0.102,0.010,0.005,0.022,0.046,0.027,0.0,...,0.00,0.0,0.367,8.7,0.0,0.10,0.0,0.5,74.91,0.15
"Pineapple, fresh",0.000,0.000,0.008,0.084,0.008,0.039,0.287,0.115,0.008,0.0,...,0.00,0.0,0.090,133.0,0.0,0.75,0.0,1640.0,87.71,1.07


In [25]:

import pandas as pd
import numpy as np
import requests
import time

# --- INPUTS ---
SHEET_CSV_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQwOKtoqUNXV0ag-KZsNvOureqjLz3H1BFwPoLuyEhdZi5_kvp2h-KPvc40VoziXBPjviI62Xl1oOdA/pub?output=csv"
USDA_API_KEY = "QgvQ1zpvzMyEORmbXPf5d5gTVzOONUCxzY1SR8a9"

FOOD_COL = "Food"     # from your screenshot
PRICE_COL = "Price"   # not used here, but kept
OUT_PATH = "foods_with_fcd_id.csv"

# --- USDA FDC SEARCH ---
SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"

# tweak this if you want different matching behavior
PREFERRED_DATA_TYPES = ["Foundation", "SR Legacy", "Survey (FNDDS)", "Branded"]  # priority order
PAGE_SIZE = 5
SLEEP_SEC = 0.15  # be nice to the API

def search_fdc(food_name: str, session: requests.Session):
    """
    Returns (fdcId, description, dataType) for best match, or (None, None, None)
    """
    if not food_name or not str(food_name).strip():
        return None, None, None

    payload = {
        "query": str(food_name),
        "pageSize": PAGE_SIZE,
        "pageNumber": 1,
        # you can uncomment to restrict:
        # "dataType": ["Foundation", "SR Legacy", "Survey (FNDDS)"]
    }

    r = session.post(SEARCH_URL, params={"api_key": USDA_API_KEY}, json=payload, timeout=30)
    r.raise_for_status()
    js = r.json()
    foods = js.get("foods", []) or []
    if not foods:
        return None, None, None

    # pick best match:
    # 1) first result with preferred datatype, else
    # 2) first result overall
    for dt in PREFERRED_DATA_TYPES:
        for f in foods:
            if f.get("dataType") == dt:
                return f.get("fdcId"), f.get("description"), f.get("dataType")

    f = foods[0]
    return f.get("fdcId"), f.get("description"), f.get("dataType")


# --- RUN ---
foods_df = pd.read_csv(SHEET_CSV_URL)

if FOOD_COL not in foods_df.columns:
    raise ValueError(f"Couldn't find '{FOOD_COL}' column. Columns are: {list(foods_df.columns)}")

# if FCD_id already exists, keep it; else create it
if "FCD_id" not in foods_df.columns:
    foods_df["FCD_id"] = np.nan

# add debug columns (optional but helpful)
if "FDC_match" not in foods_df.columns:
    foods_df["FDC_match"] = ""
if "FDC_type" not in foods_df.columns:
    foods_df["FDC_type"] = ""

session = requests.Session()
cache = {}  # food_name -> (fdcId, desc, dtype)

filled = 0
for i, name in enumerate(foods_df[FOOD_COL].astype(str).tolist()):
    if pd.notna(foods_df.loc[i, "FCD_id"]):
        continue  # already filled

    key = name.strip().lower()
    if key in cache:
        fdc_id, desc, dtype = cache[key]
    else:
        fdc_id, desc, dtype = search_fdc(name, session)
        cache[key] = (fdc_id, desc, dtype)
        time.sleep(SLEEP_SEC)

    if fdc_id is not None:
        foods_df.loc[i, "FCD_id"] = int(fdc_id)
        foods_df.loc[i, "FDC_match"] = desc or ""
        foods_df.loc[i, "FDC_type"] = dtype or ""
        filled += 1

print(f"✅ filled {filled} fdcIds")
missing = foods_df["FCD_id"].isna().sum()
print(f"still missing: {missing}")

foods_df.to_csv(OUT_PATH, index=False)
print("saved:", OUT_PATH)

✅ filled 0 fdcIds
still missing: 0
saved: foods_with_fcd_id.csv
